# Laboratorio 02.2 — Zero-shot y few-shot

Compararemos instrucciones sin ejemplos con instrucciones que incluyen demostraciones. Mantendremos fija la consulta de prueba para aislar el efecto de los ejemplos en el prompt.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Diferenciar zero-shot de few-shot por el contenido del prompt.
- Crear ejemplos demostrativos alineados con una tarea y una etiqueta.
- Medir el costo de contexto aproximado que añaden los ejemplos.
- Preparar una comparación opcional con un límite explícito de llamadas.

## 2. Conceptos

Zero-shot describe la tarea sin ejemplos etiquetados dentro del prompt. Few-shot agrega demostraciones de entradas y salidas para señalar el patrón esperado. Los ejemplos consumen contexto y pueden introducir sesgos o contradicciones; por eso deben ser representativos y consistentes. Una sola consulta no basta para concluir que una estrategia sea mejor.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Instruccion fija] --> C[Constructor zero-shot]
    B[Ejemplos etiquetados] --> D[Constructor few-shot]
    C --> E[Consulta de prueba fija]
    D --> E
    E --> F[Modelo opcional]
    F --> G[Exact match y latencia]
```

## 4. Preparación del entorno

Usamos pandas para organizar los experimentos y la fábrica multi-proveedor para la ruta opcional. No se envía ninguna consulta con la configuración inicial.

In [ ]:
from time import perf_counter

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.models import get_llm

raiz_curso = preparar_entorno()

## 5. Definición del problema

Clasificaremos consultas breves de soporte con las etiquetas `envios`, `devoluciones` y `pagos`. Las demostraciones son datos sintéticos del laboratorio. La consulta de prueba se mantiene fuera de los ejemplos para reducir contaminación directa.

## 6. Línea base

La línea base es el prompt zero-shot: tarea, etiquetas y formato esperado, sin demostraciones. Lo construimos una vez y lo inspeccionamos antes de considerar una llamada.

In [ ]:
instruccion = (
    'Clasifica la consulta en una sola etiqueta: envios, devoluciones o pagos. '
    'Devuelve solo la etiqueta, en minusculas.'
)
consulta_prueba = 'Pagué dos veces el mismo pedido.'
ejemplos = [
    {'consulta': 'Mi paquete no ha llegado.', 'etiqueta': 'envios'},
    {'consulta': 'Necesito devolver el artículo.', 'etiqueta': 'devoluciones'},
    {'consulta': 'El cobro aparece duplicado.', 'etiqueta': 'pagos'},
]

def construir_zero_shot(tarea: str, consulta: str) -> str:
    """Construye el prompt de referencia sin demostraciones."""
    return f'{tarea}\n\nConsulta: {consulta}\nEtiqueta:'

prompt_zero = construir_zero_shot(instruccion, consulta_prueba)
print(prompt_zero)

## 7. Implementación

El constructor few-shot añade pares de ejemplo en un formato repetible y separa la consulta objetivo. Observa que solo cambia la presencia de las demostraciones; la tarea y la consulta siguen constantes.

In [ ]:
def construir_few_shot(tarea: str, demostraciones: list[dict[str, str]], consulta: str) -> str:
    """Agrega ejemplos etiquetados antes de la consulta objetivo."""
    bloques = [tarea, 'Ejemplos:']
    for ejemplo in demostraciones:
        bloques.append(f"Consulta: {ejemplo['consulta']}\nEtiqueta: {ejemplo['etiqueta']}")
    bloques.append(f'Consulta: {consulta}\nEtiqueta:')
    return '\n\n'.join(bloques)

prompt_few = construir_few_shot(instruccion, ejemplos, consulta_prueba)
print(prompt_few)

## 8. Experimentos

Comparamos las dos estrategias con la misma consulta y medimos longitud aproximada en palabras. No interpretamos ese valor como tokens; un tokenizer real puede segmentar de otra manera.

In [ ]:
experimentos = pd.DataFrame([
    {'estrategia': 'zero-shot', 'ejemplos': 0, 'palabras_aprox': len(prompt_zero.split()), 'prompt': prompt_zero},
    {'estrategia': 'few-shot', 'ejemplos': len(ejemplos), 'palabras_aprox': len(prompt_few.split()), 'prompt': prompt_few},
])
display(experimentos[['estrategia', 'ejemplos', 'palabras_aprox']])
experimentos.to_csv(raiz_curso / 'outputs' / '02_02_shot_comparison.csv', index=False)

EJECUTAR_COMPARACION = False
PROVEEDOR = 'ollama'
LLAMADAS_MAXIMAS = len(experimentos)
print(f'Para ejecutar ambas estrategias se requieren como maximo {LLAMADAS_MAXIMAS} llamadas.')
resultados_modelo = []
if EJECUTAR_COMPARACION:
    modelo = get_llm(PROVEEDOR)
    for fila in experimentos.to_dict('records'):
        inicio = perf_counter()
        respuesta = modelo.invoke(fila['prompt'])
        resultados_modelo.append({
            'estrategia': fila['estrategia'],
            'salida': str(respuesta.content).strip().casefold(),
            'latencia_segundos': perf_counter() - inicio,
        })

## 9. Evaluación

Sin llamada real registramos cero respuestas, no un resultado inventado. Si se habilita la comparación, exact match mide coincidencia con la etiqueta de referencia para esta consulta; hace falta un conjunto más amplio para estimar rendimiento general.

In [ ]:
etiqueta_esperada = 'pagos'
if resultados_modelo:
    tabla_resultados = pd.DataFrame(resultados_modelo)
    tabla_resultados['exact_match'] = tabla_resultados['salida'] == etiqueta_esperada
else:
    tabla_resultados = pd.DataFrame(columns=['estrategia', 'salida', 'latencia_segundos', 'exact_match'])
metricas = pd.DataFrame([{
    'llamadas_realizadas': len(tabla_resultados),
    'exact_match_medio': tabla_resultados['exact_match'].mean() if len(tabla_resultados) else None,
}])
display(tabla_resultados)
display(metricas)
if len(tabla_resultados):
    tabla_resultados.to_csv(raiz_curso / 'outputs' / '02_02_model_results.csv', index=False)

## 10. Discusión

Few-shot aporta señales de formato y decisión, pero cuesta contexto y ejemplos mal elegidos pueden empeorar el comportamiento. Un cambio entre dos respuestas puede deberse a variación del modelo; para comparar con rigor se requieren varias consultas, registro de configuración y evaluación separada.

## 11. Ejercicios

1. Compara uno, tres y cinco ejemplos manteniendo fija la consulta.
2. Sustituye un ejemplo consistente por uno contradictorio y formula una hipótesis.
3. Construye un conjunto de validación con etiquetas equilibradas.
4. Cambia el orden de los ejemplos y mide si la salida varía.

## 12. Desafío

Diseña una búsqueda de ejemplos few-shot que elija demostraciones relevantes desde un conjunto separado. Compara costo de contexto y accuracy frente a ejemplos elegidos al azar, con límite explícito de llamadas.

## 13. Ideas clave

- Zero-shot especifica la tarea sin demostraciones; few-shot añade pares etiquetados.
- Ejemplos ocupan contexto y pueden introducir sesgos.
- Mantén fija la consulta al comparar estrategias.
- Un solo acierto no demuestra superioridad general.